# Imports

In [ ]:
import sys
sys.path.append("../") # local utils.py path
import numpy as np
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
import cv2
import sys
import utils
import gc
import time
import importlib
import torch
importlib.reload(utils)
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from czifile import CziFile
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
from matplotlib.patches import Circle
import ast
from matplotlib.colors import ListedColormap, Normalize

In [ ]:
utils.print_version()

## Confused by the parameter names? Try *help(utils.function_name)*

**Example**

In [ ]:
help(utils.plot_keypoint_tracking)

# Load Files

1. all_channel := single czi image with all color channels
2. Tm Probe 1 (Tmp1) := czi image series of probe color 1 (e.g. HEX, ROX, CY5) Order does not matter
3. Tm Probe 2 (Tmp2) := czi image series of probe color 2 (e.g. HEX, ROX, CY5) Order does not matter
4. Tm Barcode_Target (Tmb) := czi image series of EvaGreen Channel

In [ ]:
importlib.reload(utils)

In [ ]:
parent_dir = ""

In [ ]:
utils.load_directory(parent_dir, extension="czi", local_scope=locals(), 
                               file_path_variable_list=["all_channel_file_path",
                                                        "Tmp1_file_path",
                                                        "Tmp2_file_path",
                                                        "Tmb_file_path"], font_size="50px")

In [ ]:
tmP_1_file_path = Tmp1_file_path
tmP_2_file_path = Tmp2_file_path
tmB_T_file_path = Tmb_file_path

In [ ]:
tmP_1_save_filename = utils.parse_filename(tmP_1_file_path)
tmP_2_save_filename = utils.parse_filename(tmP_2_file_path)
tmB_T_save_filename = utils.parse_filename(tmB_T_file_path)

In [ ]:
if 'tmP_1_save_filename' in locals():
    print(tmP_1_save_filename)
if 'tmP_2_save_filename' in locals():
    print(tmP_2_save_filename)
if 'tmB_T_save_filename' in locals():
    print(tmB_T_save_filename)

In [ ]:
all_file_paths = [tmP_1_file_path, tmP_2_file_path, tmB_T_file_path]

all_file_paths = [tmP_1_file_path, tmB_T_file_path]

# Set Constants

1. k0: int, odd numbers (in pixels) := kernel size for the first round of gaussian convolution
2. k1: int, odd numbers (in pixels) := kernel size for the second round of convolution (locality size) in pixels
3. pixel_range: int (in pixels) := diameter in which we use to extract fluorescence values
4. merge_threshold: int (in pixels) := in multiple rounds of positive well detection, the max distance between two detected centers to be considered to be of the same well
5. eps: int (in pixels) := max search range between consecutive frames during alignment process

In [ ]:
# For 32x mag: set k0=15-17; for 16x mag: set k0=9
k0 = 15
k1 = 15
pixel_range = 17 
merge_threshold=17 # close but smaller than or equals to pixel range
eps = 12

# Image Processing and Analysis

#### Determine order of processing based on time of image acquisition

In [ ]:
metadata, n_channels, channel_names, excitation_wavelengths, exposure_times, laser_intensity, colors = utils.retrieve_czi_metadata(all_channel_file_path)

In [ ]:
importlib.reload(utils)

In [ ]:
img_stack, divider_idxs, ordered_channel_names, channel_idx_dict = utils.determine_processing_order(all_file_paths, 
                                                                        override_name_list=["Cy5", "HEX", "EGFP"])  # Note: Adjust based on actual channels present

img_series_lengths = divider_idxs.copy()
img_series_lengths[1:]-= img_series_lengths[:-1].copy()

### Run Detection on Full Array

In [ ]:
start_time = time.time()
all_pos_seq = utils.generate_pos_seq_new_no_tile(img_stack, k0=k0, k1=k1, 
                                 hist_threshold=0, get_negative=False, enhance=True, gamma = 0.001,
                                                 bkg_correct_radius=5, bkg_correct_sigma=20)
end_time = time.time()
print("Runtime:", np.round(end_time - start_time, 2), "seconds")
detection_runtime = np.round(end_time - start_time, 2)

In [ ]:
first_frame_pos = utils.generate_pos_seq_new_no_tile(img_stack[0:1,:,:], k0=k0, k1=k1, 
                                 hist_threshold=0, get_negative=False, enhance=True, gamma = 0.001, bkg_correct_radius=1, bkg_correct_sigma=20)[0]

In [ ]:
# QC Check
# Expected len = number of frames; Expected shape = (number of positives, 2)
print(f"Number of frames: {len(all_pos_seq)}; Actual Shape: {all_pos_seq[0].shape}")

### Alignment

In [ ]:
start_time = time.time()
all_pts_seq = utils.track_keypoints_multi_channel(all_pos_seq, eps, divider_idxs) # epsilon = Search range (pixels)
# all_pts_seq = utils.track_keypoints(all_pos_seq, eps)
end_time = time.time()
print("Runtime:", np.round(end_time - start_time, 2), "seconds")
alignment_runtime = np.round(end_time - start_time, 7)

### Filter Edge Points

In [ ]:
filtered_pts_seq, invalid_idxs, valid_idxs = utils.filter_keypoints(img_stack, all_pts_seq, R = 9)
# all_pos_seq = [frame[valid_idxs] for frame in all_pos_seq]
all_pos_seq_one = [all_pos_seq[0][valid_idxs]]

In [ ]:
# Generate un-aligned results for QC
frame_one = all_pos_seq_one[0]
unaligned = frame_one[:, np.newaxis, :]
unaligned = np.tile(unaligned, (1, len(all_pos_seq), 1))
unaligned.shape

### Alignment QC

In [ ]:
channel_idx_dict

In [ ]:
utils.plot_keypoint_tracking(img_stack, unaligned, keypoint_index=2500, m=200, 
                             alignment_correction=filtered_pts_seq, save_video=False, video_filename="tracking.mp4")

In [ ]:
raw_fluorescence_vals = utils.generate_fluorescence_vs_time(img_arr = img_stack,
                                                 pts_seq = all_pts_seq[valid_idxs],
                                                 pix_range=pixel_range,
                                                 filter = "None")

In [ ]:
raw_fluorescence_vals.shape

In [ ]:
# SKIP If no probes
probe_first_frame_intensities = []
for i, idx in enumerate(divider_idxs[:2]):
    img_to_process = np.expand_dims(utils.gaussian_background_correction(img_stack[idx]),axis=0)
    pts_seq_to_process = all_pts_seq[valid_idxs][:, idx:idx+1, :]
    print(pts_seq_to_process.shape)
    corrcted_optical_fluorescence_vals=utils.generate_fluorescence_vs_time(img_arr = img_to_process,
                                                 pts_seq = pts_seq_to_process,
                                                 pix_range=pixel_range,
                                                 filter = "None", gaussian = False)
    probe_first_frame_intensities.append(corrcted_optical_fluorescence_vals)

In [ ]:
# QC Check
# Expected SNR >= 20 for 32x data
# Expected SNR >= 14 for 20x data
utils.snr_moving_avg(raw_fluorescence_vals, window=5)

## Save All Data

In [ ]:
importlib.reload(utils)

In [ ]:
data_to_save = {
    "all_pos_seq": all_pos_seq,
    "all_pts_seq": all_pts_seq,
    "filtered_pts_seq": filtered_pts_seq,
    "invalid_idxs": invalid_idxs,
    "valid_idxs": valid_idxs,
    "all_pos_seq_one": all_pos_seq_one,
    "frame_one": frame_one,
    "unaligned": unaligned,
    "channel_idx_dict": channel_idx_dict,
    "raw_fluorescence_vals": raw_fluorescence_vals,
    "probe_first_frame_intensities": probe_first_frame_intensities
}

In [ ]:
utils.save_data(data_to_save)

# SKIP TO Analysis

In [ ]:
parent_dir = ""

In [ ]:
utils.load_directory(parent_dir, extension="czi", local_scope=locals(), 
                               file_path_variable_list=["all_channel_file_path",
                                                        "tmP_1_file_path",
                                                        "tmP_2_file_path",
                                                        "tmB_T_file_path"], font_size="50px")

In [ ]:
tmP_1_save_filename = utils.parse_filename(tmP_1_file_path)
tmP_2_save_filename = utils.parse_filename(tmP_2_file_path)
tmB_T_save_filename = utils.parse_filename(tmB_T_file_path)

In [ ]:
if 'tmP_1_save_filename' in locals():
    print(tmP_1_save_filename)
if 'tmP_2_save_filename' in locals():
    print(tmP_2_save_filename)
if 'tmB_T_save_filename' in locals():
    print(tmB_T_save_filename)

In [ ]:
all_file_paths = [tmP_1_file_path, tmP_2_file_path, tmB_T_file_path]

In [ ]:
# For 32x mag: set k0=15-17; for 16x mag: set k0=9
k0 = 15
k1 = 15
pixel_range = 17 
merge_threshold=17 # close but smaller than or equals to pixel range
eps = 12

In [ ]:
metadata, n_channels, channel_names, excitation_wavelengths, exposure_times, laser_intensity, colors = utils.retrieve_czi_metadata(all_channel_file_path)

In [ ]:
img_stack, divider_idxs, ordered_channel_names, channel_idx_dict = utils.determine_processing_order(all_file_paths, 
                                                                        override_name_list=["Cy5", "HEX", "EGFP"])
img_series_lengths = divider_idxs.copy()
img_series_lengths[1:]-= img_series_lengths[:-1].copy()

In [ ]:
try:
    loaded_data = utils.load_data()
    all_pos_seq = loaded_data["all_pos_seq"]
    all_pts_seq = loaded_data["all_pts_seq"]
    filtered_pts_seq = loaded_data["filtered_pts_seq"]
    invalid_idxs = loaded_data["invalid_idxs"]
    valid_idxs = loaded_data["valid_idxs"]
    all_pos_seq_one = loaded_data["all_pos_seq_one"]
    frame_one = loaded_data["frame_one"]
    unaligned = loaded_data["unaligned"]
    channel_idx_dict = loaded_data["channel_idx_dict"]
    raw_fluorescence_vals = loaded_data["raw_fluorescence_vals"]
    probe_first_frame_intensities = loaded_data["probe_first_frame_intensities"]
    
    print("Data loaded successfully! Skip to analysis section.")
    
except FileNotFoundError:
    print("No saved data found. Running data collection...")

## Create Positive-Rain-Negative Paritions (based on EvaGreen Channel)

In [ ]:
importlib.reload(utils)

In [ ]:
tmB_T_first_frame = img_stack[channel_idx_dict["EGFP"][0]]
tmB_T_first_frame = utils.gaussian_background_correction_div(tmB_T_first_frame, radius=50)
positive_idxs, rain_idxs, negative_idxs, positive_pos, raind_pos, negative_pos=utils.define_the_rain(tmB_T_first_frame, 
                                    [all_pts_seq[:, channel_idx_dict["EGFP"][0], :][valid_idxs]], n_SD = 2.7, rain=False)

In [ ]:
utils.plot_labeled_img_multi(tmB_T_first_frame, positive_pos, raind_pos, channel_names, 
                       excitation_wavelengths, exposure_times, laser_intensity, 
                       def_the_rain_sd=1., gamma=1., marker = "circ") 
                        # gamma controls the displayed image contrast. larger --> higher contrast; gamm=1 --> Image unchaged
                

In [ ]:
bkg_strength_list = []
temp = np.insert(divider_idxs, 0,0)
# temp = [269, 269, 600, 931]
for idx in range(0, len(divider_idxs)):
    print(np.expand_dims(img_stack[temp[idx]],axis=0).shape)
    print(np.expand_dims(all_pos_seq[0][negative_idxs], axis=1).shape)
    bkg_fluorescence_vals = np.median(utils.generate_fluorescence_vs_time(img_arr = np.expand_dims(img_stack[temp[idx]],axis=0),
      pts_seq =filtered_pts_seq[:, temp[idx]:temp[idx]+1, :][negative_idxs],                                                                    
                                                 pix_range=pixel_range,
                                                 filter = "None"))
    positive_fluorescence_vals = np.median(utils.generate_fluorescence_vs_time(img_arr = np.expand_dims(img_stack[temp[idx]],axis=0),
      pts_seq =filtered_pts_seq[:, temp[idx]:temp[idx]+1, :][positive_idxs],                                                                       
                                                 pix_range=pixel_range,
                                                 filter = "None"))
    # print(bkg_fluorescence_vals, positive_fluorescence_vals)
    tot = bkg_fluorescence_vals+positive_fluorescence_vals
    print(bkg_fluorescence_vals/positive_fluorescence_vals)
    
    bkg_relative_strength = bkg_fluorescence_vals/tot
    bkg_strength_list.append(bkg_relative_strength)
    

In [ ]:
print(bkg_strength_list)

# Split signals back to corresponding channels

In [ ]:
print(f"""Reminder that the channesl are in the following order: {ordered_channel_names}.
The corresponding cutoff indices are: {divider_idxs}.""")

In [ ]:
tmB_T_raw_fluorescence_vals = raw_fluorescence_vals[:, channel_idx_dict["EGFP"][0]:channel_idx_dict["EGFP"][1]]
print(tmB_T_raw_fluorescence_vals.shape)
tmP_1_raw_fluorescence_vals_uncut = raw_fluorescence_vals[:, channel_idx_dict["Cy5"][0]:channel_idx_dict["Cy5"][1]]
print(tmP_1_raw_fluorescence_vals_uncut.shape)
tmP_2_raw_fluorescence_vals_uncut = raw_fluorescence_vals[:, channel_idx_dict["HEX"][0]:channel_idx_dict["HEX"][1]]
print(tmP_2_raw_fluorescence_vals_uncut.shape)

In [ ]:
fig_width = 15
fig_height = 10
plt.figure(figsize=(fig_width, fig_height))
plt.subplot(2, 2, 1) 
for signal in tmP_1_raw_fluorescence_vals_uncut:
    plt.plot(signal)
plt.xlabel("Frames")
plt.ylabel("Fluorescence Value") 
plt.title("All Tm Probe 1")

plt.subplot(2, 2, 2)
for signal in tmP_2_raw_fluorescence_vals_uncut:
    plt.plot(signal)
plt.xlabel("Frames")
plt.ylabel("Fluorescence Value") 
plt.title("All Tm Probe 2")


plt.subplot(2, 1, 2)
for signal in tmB_T_raw_fluorescence_vals:
    plt.plot(signal)
plt.xlabel("Frames")
plt.ylabel("Fluorescence Value") 
plt.title("All Tm Barcode_Target")

plt.tight_layout()
plt.show()


# Tm Barcode_Target (in EvaGreen)

## Set Constants

1. tmB_T_inital_T: float (in Celcius) := Inital heating plate temperature for Tm Barcode_Target
2. tmB_T_final_T: float (in Celcius) := Final heating plate temperature for Tm Barcode_Target
3. tmB_T_heating_rate: float (in Celcius/min) := Heating plate temperature rate of change
4. tmB_T_img_series_gap_time: float (in Seconds) := Time gap between each frame

In [ ]:
tmB_T_inital_T = 70.0
tmB_T_final_T = 90.0
tmB_T_heating_rate = 5.0
tmB_T_img_series_gap_time = 1

In [ ]:
img_series_lengths

In [ ]:
channel_idx_dict

In [ ]:
tmB_T_xticks = np.arange(0, channel_idx_dict['EGFP'][1]-channel_idx_dict['EGFP'][0], step=25)
tmB_T_temps = np.zeros(tmB_T_xticks.shape)
for i, frame in enumerate(tmB_T_xticks):
    tmB_T_temps[i] = utils.compute_Tm(initial_T=tmB_T_inital_T, peak_frame_idx=frame,
                                         rate_per_min=tmB_T_heating_rate, 
                                         exposure_in_sec=tmB_T_img_series_gap_time,
                                        max=tmB_T_final_T)
tmB_T_temps = np.round(tmB_T_temps, 1)
tmB_T_temps

In [ ]:
# tmB_T_raw_fluorescence_vals = tmB_T_raw_fluorescence_vals[:, :]

## Positive, Rain, and Negative Paritions Visualization

In [ ]:
fig_width = 15
fig_height = 10
plt.figure(figsize=(fig_width, fig_height))
plt.subplot(2, 2, 1) 
for signal in tmB_T_raw_fluorescence_vals[rain_idxs]:
    plt.plot(signal)
plt.xlabel("Frames")
plt.ylabel("Fluorescence Value") 
plt.title("Rain Parition")

plt.subplot(2, 2, 2)
for signal in tmB_T_raw_fluorescence_vals[negative_idxs]:
    plt.plot(signal)
plt.xlabel("Frames")
plt.ylabel("Fluorescence Value") 
plt.title("Negative Parition")

plt.subplot(2, 1, 2)
for signal in tmB_T_raw_fluorescence_vals[positive_idxs]:
    plt.plot(signal)
plt.xlabel("Frames")
plt.ylabel("Fluorescence Value") 
plt.title("Positive Partition")
plt.tight_layout()
plt.show()


In [ ]:
tmB_T_data = tmB_T_raw_fluorescence_vals[positive_idxs]

### Background Subtraction (updated bkg subtraction method 01/31/25)

In [ ]:
normalized_tmB_T = utils.min_max_normalize(tmB_T_data)

In [ ]:
tmB_T_raw_bkg = tmB_T_raw_fluorescence_vals[negative_idxs]
# tmB_T_bkg = utils.min_max_normalize(np.median(tmB_T_raw_bkg, axis=0))
# tmB_T_bkg_subtracted = normalized_tmB_T-bkg_strength_list[0]*tmB_T_bkg
tmB_T_bkg_subtracted = utils.min_max_normalize(utils.wittwer_background_subtract(tmB_T_data, TL_idx=8, TR_idx=255, eps = 5, plot = False))  # No need to specify background strength; Adjust TL/TR accordingly
np.save("bkg_subtracted_raw_"+tmB_T_save_filename, tmB_T_bkg_subtracted)

In [ ]:
# x_values = np.linspace(70, 90, 271)

## Obtain -dF/dT Melting Curve

In [ ]:
tmB_T_mcs_noDeriv = utils.min_max_normalize(1*utils.savgol_filter(tmB_T_bkg_subtracted, window_length=45, 
                                                                      polyorder=2, deriv=0,mode='nearest'), use_global_min_max=True)
unfiltered_tmB_T_mcs_deriv = utils.min_max_normalize(-1*utils.savgol_filter(tmB_T_bkg_subtracted, window_length=45, 
                                                                     polyorder=2, deriv=1,mode='nearest'), use_global_min_max=True)

In [ ]:
tmB_T_bkg_deriv = utils.min_max_normalize(-1*utils.savgol_filter(tmB_T_raw_bkg, window_length=45, 
                                                                     polyorder=2, deriv=1,mode='nearest'), use_global_min_max=True)

In [ ]:
fig_width = 15
fig_height = 5
plt.figure(figsize=(fig_width, fig_height))

plt.subplot(1, 2, 1) 
for signal in tmB_T_bkg_subtracted:
    plt.plot(signal)
plt.xticks(ticks=tmB_T_xticks, labels=tmB_T_temps)
plt.xlabel("Temperature (C)")
plt.ylabel("Fluorescence Value")
plt.title("Normalized Tm Barcode_Target (Background Subtracted)")

plt.subplot(1, 2, 2)
for signal in tmB_T_mcs_noDeriv:
    plt.plot(signal)
plt.xticks(ticks=tmB_T_xticks, labels=tmB_T_temps)
plt.xlabel("Temperature (C)")
plt.ylabel("Normalized Fluorescence Value")
plt.title("Smoothed Tm Barcode_Target (Background Subtracted)")
plt.tight_layout()
plt.show()


In [ ]:
fig_width = 15
fig_height = 5
plt.figure(figsize=(fig_width, fig_height))

plt.subplot(1, 2, 1) 
for signal in tmB_T_mcs_noDeriv:
    plt.plot(signal)
plt.xticks(ticks=tmB_T_xticks, labels=tmB_T_temps)
plt.xlabel("Temperature (C)")
plt.ylabel("Normalized Fluorescence Value")
plt.title("Smoothed Tm Barcode_Target (Background Subtracted)")

plt.subplot(1, 2, 2)
for signal in unfiltered_tmB_T_mcs_deriv:
    plt.plot(signal)
plt.xticks(ticks=tmB_T_xticks, labels=tmB_T_temps)
plt.xlabel("Temperature (C)")
plt.ylabel("Normalized -dF/dT")
plt.title("Normalized Tm Barcode_Target (Background Subtracted)")
plt.tight_layout()
plt.show()


In [ ]:
tmB_T_anomaly_mask = utils.interactive_anomaly_filtering(normalized_tmB_T, mode="iso")

In [ ]:
clean_tmB_T_mcs_deriv_mask = tmB_T_anomaly_mask['normal_mask']
abnormal_tmB_T_mask = tmB_T_anomaly_mask['anomaly_mask']

tmB_T_mcs_deriv = unfiltered_tmB_T_mcs_deriv[clean_tmB_T_mcs_deriv_mask]
abnormal_tmB_T = unfiltered_tmB_T_mcs_deriv[abnormal_tmB_T_mask]

In [ ]:
normalized_tmB_T = normalized_tmB_T[clean_tmB_T_mcs_deriv_mask]
clean_tmB_T_bkg_subtracted = tmB_T_bkg_subtracted[clean_tmB_T_mcs_deriv_mask]

In [ ]:
tmB_T_bkg_subtracted = tmB_T_bkg_subtracted[clean_tmB_T_mcs_deriv_mask]

In [ ]:
tmB_T_mcs_deriv.shape

In [ ]:
tmB_T_bkg_subtracted.shape

# Melting temperature determination

In [ ]:
importlib.reload(utils)

### Level 1: Derivative Window Size = 45 (Frames) --> About 3.75 (C)

In [ ]:
tmB_T_noise_floor, tmB_T_noise_floor_params, upper_confid = utils.get_noise_floor(normalized_array = tmB_T_mcs_deriv, before_nth_frame=20, 
                                              plot=True, tm_temps=tmB_T_temps, tm_xticks=tmB_T_xticks, n_SD=1, compute_SD=True)

In [ ]:
importlib.reload(utils)

In [ ]:
utils.generate_variable_threshold(signal_length=271,low_threshold_segments=[(0,270)],
                                                           low_threshold_value=0,
                                                           high_threshold_value=1).shape

In [ ]:
tmB_Ts_lvl1, one_peaks_lvl1, two_peaks_lvl1, more_than_two_peaks_lvl1, tmB_T_heights, Tm_indices,new_tmB_T_nFloor, fhbl  = utils.get_Tm(mcs = tmB_T_mcs_deriv, k_peaks=2, 
                                                        first_frame_T=tmB_T_inital_T, heating_rate_per_min=tmB_T_heating_rate,
                                                        img_series_gap_time = tmB_T_img_series_gap_time, noise_floor = upper_confid,
                                    tm_temps=tmB_T_temps, tm_xticks=tmB_T_xticks, max_temp=tmB_T_final_T, widths=(15),                 
                                                        noise_floor_params=tmB_T_noise_floor_params, return_new_noise_floor=True,
                                                                    plot=True, weight = 0.5, height_tolerance=5, use_upper_conf=False,
                                                        ignore_first_nFrames=30)

In [ ]:
tmB_Ts_lvl1, one_peaks_lvl1, two_peaks_lvl1, more_than_two_peaks_lvl1, tmB_T_heights, Tm_indices,new_tmB_T_nFloor, fhbl  = utils.get_Tm(mcs = tmB_T_mcs_deriv, k_peaks=2, 
                                                        first_frame_T=tmB_T_inital_T, heating_rate_per_min=tmB_T_heating_rate,
                                                        img_series_gap_time = tmB_T_img_series_gap_time, noise_floor = upper_confid,
                                    tm_temps=tmB_T_temps, tm_xticks=tmB_T_xticks, max_temp=tmB_T_final_T, widths=(15),                 
                                                        noise_floor_params=tmB_T_noise_floor_params, return_new_noise_floor=True,
                                                                    plot=True, weight = 0.5, height_tolerance=5, use_upper_conf=False)

### Level 1 Partition Visualization

In [ ]:
utils.visualize_melt_curve_partitions(global_tm_mcs_deriv = tmB_T_mcs_deriv,
                                     one_peaks=one_peaks_lvl1, two_peaks=two_peaks_lvl1, more_than_two_peaks=more_than_two_peaks_lvl1,
                                     global_tm_xticks=tmB_T_xticks, global_tm_temps=tmB_T_temps)

In [ ]:
level1_MoreThanTwo_data = tmB_T_mcs_deriv[more_than_two_peaks_lvl1]
level1_MoreThanTwo_pos = positive_pos[more_than_two_peaks_lvl1]
level1_MoreThanTwo_all = np.hstack([level1_MoreThanTwo_pos, level1_MoreThanTwo_data])
np.save("level1_MoreThanTwo.npy", level1_MoreThanTwo_all)
# Note: first two columns contain their position information

In [ ]:
len(one_peaks_lvl1), len(two_peaks_lvl1), len(more_than_two_peaks_lvl1)

In [ ]:
tmB_Ts_lvl1.shape

### Level 2/3: Derivative Window Size = 35 (Frames) --> About 2.6 (C)

In [ ]:
reduced_window_derive = utils.min_max_normalize(-1*utils.savgol_filter(tmB_T_bkg_subtracted, window_length=35, 
                                                                     polyorder=2, deriv=1,mode='nearest'), use_global_min_max=True)

#### Re-processing previously considered two-peaks

In [ ]:
tmB_Ts_lvl2, one_peaks_lvl2, two_peaks_lvl2, more_than_two_peaks_lvl2, _, _, new_tmB_T_nFloor, _ = utils.get_Tm_lvl2(global_tm_bkg_subtracted = tmB_T_bkg_subtracted, 
                                                        one_peak_idxs= two_peaks_lvl1, k_peaks=2, 
                                                        first_frame_T=tmB_T_inital_T, heating_rate_per_min=tmB_T_heating_rate,
                                                        img_series_gap_time = tmB_T_img_series_gap_time,
                                                        return_new_noise_floor=True, widths = 10,
                                                        tm_temps=tmB_T_temps, tm_xticks=tmB_T_xticks, max_temp=tmB_T_final_T,
                                                        plot=True, new_window_length=31, weight = 0.5, n_SD=1, height_tolerance=0.05, use_upper_conf=False)

In [ ]:
utils.visualize_melt_curve_partitions(global_tm_mcs_deriv = reduced_window_derive,
                                     one_peaks=one_peaks_lvl2, two_peaks=two_peaks_lvl2, more_than_two_peaks=more_than_two_peaks_lvl2,
                                     global_tm_xticks=tmB_T_xticks, global_tm_temps=tmB_T_temps)

In [ ]:
level2_MoreThanTwo_data = tmB_T_mcs_deriv[more_than_two_peaks_lvl2]
level2_MoreThanTwo_pos = positive_pos[more_than_two_peaks_lvl2]
level2_MoreThanTwo_all = np.hstack([level2_MoreThanTwo_pos, level2_MoreThanTwo_data])
np.save("level2_MoreThanTwo.npy", level2_MoreThanTwo_all)
# Note: first two columns contain their position information

#### Re-processing previously considered one-peaks

In [ ]:
tmB_Ts_lvl3, one_peaks_lvl3, two_peaks_lvl3, more_than_two_peaks_lvl3, _, _, _, _ = utils.get_Tm_lvl2(global_tm_bkg_subtracted = tmB_T_bkg_subtracted, 
                                                        one_peak_idxs= one_peaks_lvl1, k_peaks=2, 
                                                        first_frame_T=tmB_T_inital_T, heating_rate_per_min=tmB_T_heating_rate,
                                                        img_series_gap_time = tmB_T_img_series_gap_time,
                                                        return_new_noise_floor=True, widths = 5,
                                                        tm_temps=tmB_T_temps, tm_xticks=tmB_T_xticks, max_temp=tmB_T_final_T,
                                                        plot=True, new_window_length=31, weight = 1, n_SD=.8, height_tolerance=5., use_upper_conf=True)

In [ ]:
utils.visualize_melt_curve_partitions(global_tm_mcs_deriv = reduced_window_derive,
                                     one_peaks=one_peaks_lvl3, two_peaks=two_peaks_lvl3, more_than_two_peaks=more_than_two_peaks_lvl3,
                                     global_tm_xticks=tmB_T_xticks, global_tm_temps=tmB_T_temps)

In [ ]:
level3_MoreThanTwo_data = tmB_T_mcs_deriv[more_than_two_peaks_lvl3]
level3_MoreThanTwo_pos = positive_pos[more_than_two_peaks_lvl3]
level3_MoreThanTwo_all = np.hstack([level3_MoreThanTwo_pos, level3_MoreThanTwo_data])
np.save("level3_MoreThanTwo.npy", level3_MoreThanTwo_all)
# Note: first two columns contain their position information

In [ ]:
one_peaks = one_peaks_lvl2+two_peaks_lvl3
two_peaks = two_peaks_lvl2+two_peaks_lvl3
more_than_two_peaks = more_than_two_peaks_lvl2+more_than_two_peaks_lvl3
if tmB_Ts_lvl3.shape[0] != 0:
    tmB_Ts = np.vstack([tmB_Ts_lvl2, tmB_Ts_lvl3])
else:
    tmB_Ts = tmB_Ts_lvl2

In [ ]:
tmB_T_flc = tmB_T_bkg_subtracted[two_peaks]
tmB_T_dfc = tmB_T_mcs_deriv[two_peaks]

In [ ]:
data_df = pd.DataFrame(np.hstack([np.arange(tmB_Ts.shape[0]).reshape(-1,1), tmB_Ts]), columns=["idx","LowTm", "HighTm"])

In [ ]:
updated_data_df = utils.interactive_visual_QC(plot_dfs=[data_df],
                           external_arr_list1=[tmB_T_flc],
                           external_arr_list2=[tmB_T_dfc],
                           external_arr_list3=[reduced_window_derive[two_peaks]],
                           external_arr_list4= None,
                            plot_12_ticks = tmB_T_temps,
                            plot_34_ticks = tmB_T_temps,
                           selected_col_names=["LowTm", "HighTm"])

In [ ]:
valid_two_peak_idxs = updated_data_df[0][updated_data_df[0]["validity_status"]==1]["idx"].to_numpy().astype("int32")
two_peaks = [two_peaks[i] for i in valid_two_peak_idxs]
tmB_Ts = tmB_Ts[valid_two_peak_idxs]
tmB_T_flc = tmB_T_bkg_subtracted[two_peaks]
tmB_T_dfc = tmB_T_mcs_deriv[two_peaks]
print(len(two_peaks))

In [ ]:
tmB_Ts_lvl2.shape

In [ ]:
tmB_Ts.shape

#### Keeping data points with 2 peaks

In [ ]:
tmB_T_corrected_idxs = two_peaks

In [ ]:
plt.figure(figsize=(15, 13))
plt.scatter(tmB_Ts[:,0], tmB_Ts[:,1], marker='o', alpha=0.7, s=7)
y_min, x_min = 77, 72
y_max, x_max = 91.5, 86.5
plt.xlabel('Low Tm', fontsize=20)
plt.ylabel('High Tm', fontsize=20)
plt.xticks(np.arange(x_min, x_max, 0.5), fontsize=10)
plt.yticks(np.arange(y_min, y_max, 0.5), fontsize=10)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.grid(True)
plt.show()

In [ ]:
np.save("bkg_subtracted_raw_"+tmB_T_save_filename, tmB_T_bkg_subtracted[tmB_T_corrected_idxs])

In [ ]:
np.save("dfc_"+tmB_T_save_filename, tmB_T_mcs_deriv[tmB_T_corrected_idxs])

# Run this subsection when processing Protein data

## (Skip otherwise)

In [ ]:
# Filter by EvaGreen 2-peaks
filtered_pos = positive_pos[tmB_T_corrected_idxs]
filtered_raw_fluorescence = tmB_T_data[tmB_T_corrected_idxs]


tmB_T_df = data_df.iloc[valid_two_peak_idxs]

tmB_T_df.reset_index(inplace=True)

tmB_T_df.to_csv("all_tms.csv")
all_combined = utils.join_meta_data(tmB_T_df, filtered_pos, 
                                    filtered_raw_fluorescence)

if 'FOV_well_counts' in all_combined.columns:
    all_combined = all_combined.drop(columns=['FOV_well_counts'])

all_combined.insert(5, 'FOV_well_counts', [all_pts_seq[:, channel_idx_dict["EGFP"][0], :][valid_idxs]][0].shape[0])


if 'FOV_positive_counts' in all_combined.columns:
    all_combined = all_combined.drop(columns=['FOV_positive_counts'])

all_combined.insert(6, 'FOV_positive_counts', positive_pos.shape[0])

all_combined.to_csv(r"raw_tm_result.csv")

# 

# Run this subection when processing DNA data (No probe Tm)

## (Skip otherwise)

In [ ]:
importlib.reload(utils)

In [ ]:
# 1. Filter by Tm Barcode_Target Positive
tmP_1_data = tmP_1_raw_fluorescence_vals_uncut[positive_idxs][clean_tmB_T_mcs_deriv_mask]
tmP_1_raw_bkg = tmP_1_raw_fluorescence_vals_uncut[negative_idxs]
tmP_1_bkg = utils.savgol_filter(utils.min_max_normalize(np.median(tmP_1_raw_bkg, axis=0)),window_length=55, 
                                                             polyorder=2, deriv=0,mode='nearest')
# 2. Filter by Tm Barcode_Target 2-peak Tm
tmB_T_corrected_tmP_1_data = tmP_1_data[tmB_T_corrected_idxs]

In [ ]:
tmP_1_first_frame = img_stack[channel_idx_dict["Cy5"][1]-1]
tmP_1_first_frame = utils.gaussian_background_correction_div(tmP_1_first_frame, radius=20, sigma=30)

tmP_1_positive_idxs, _, tmP_1_negative_idxs, tmP_1_positive_pos, _, tmP_1_negative_pos, tmP_1_fluor=utils.define_the_rain(tmP_1_first_frame, 
                             [all_pts_seq[:, channel_idx_dict["Cy5"][1]-1, :][valid_idxs][positive_idxs][clean_tmB_T_mcs_deriv_mask][tmB_T_corrected_idxs]],
                            n_SD = 2, rain=False, return_positive_fluorescence=True,  adaptive=True, half_win=100)

In [ ]:
utils.plot_labeled_img_multi(tmP_1_first_frame, tmP_1_positive_pos, tmP_1_negative_pos, channel_names, 
                       excitation_wavelengths, exposure_times, laser_intensity, 
                       def_the_rain_sd=3., gamma=1., marker = "circ", i=0) 
                        # gamma controls the displayed image contrast. larger --> higher contrast; gamm=1 --> Image unchaged
                

In [ ]:
cy5_tms = tmB_Ts[tmP_1_positive_idxs]
cy5_channel_positives = pd.DataFrame(np.hstack([np.array(tmP_1_positive_idxs).reshape(-1,1), tmB_Ts[tmP_1_positive_idxs]]), columns=["idx","LowTm", "HighTm"])
cy5_channel_positives.to_csv("red.csv")
other_tms = tmB_Ts[tmP_1_negative_idxs]
other_channel_positives = pd.DataFrame(np.hstack([np.array(tmP_1_negative_idxs).reshape(-1,1), tmB_Ts[tmP_1_negative_idxs]]),  columns=["idx","LowTm", "HighTm"])
other_channel_positives.to_csv("other.csv")

In [ ]:
plt.figure(figsize=(15, 15))
plt.scatter(cy5_tms[:,0], cy5_tms[:,1], marker='o', alpha=0.7, s=7)
plt.scatter(other_tms[:,0], other_tms[:,1], marker='o', alpha=0.7, s=7)

y_min, x_min = 77, 72
y_max, x_max = 91.5, 86.5
plt.xlabel('Low Tm', fontsize=20)
plt.ylabel('High Tm', fontsize=20)
plt.xticks(np.arange(x_min, x_max, 0.5), fontsize=10)
plt.yticks(np.arange(y_min, y_max, 0.5), fontsize=10)
plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.grid(True)
plt.show()

In [ ]:
total_well_cnts = first_frame_pos.shape[0]
total_eva_positive = positive_pos.shape[0]
two_peaks_cnts = len(two_peaks) 
additional_stats = np.array([total_well_cnts, total_eva_positive, two_peaks_cnts])
np.save("additional_stats.npy", additional_stats)

In [ ]:
# Get all 2-peak positions
final_two_peak_pos = positive_pos[tmB_T_corrected_idxs]
final_two_peak_pos_df = pd.DataFrame(final_two_peak_pos, columns=["X", "Y"])
# Save color channel info
total = sorted(set(list(tmP_1_positive_idxs) + list(tmP_1_negative_idxs)))
channel_df = pd.DataFrame(index=total)
channel_df['Color'] = 'other'
channel_df.loc[tmP_1_positive_idxs, 'Color'] = 'Red'
channel_df.loc[tmP_1_negative_idxs, 'Color'] = 'Other'
# Save intensity info
intensity_df = pd.DataFrame(tmB_T_flc, columns=[f'intensity_{i}' for i in range(tmB_T_flc.shape[1])])
combined_data = pd.concat([data_df, final_two_peak_pos_df, channel_df, intensity_df], axis=1)
combined_data.to_csv("raw_tm_result.csv")
combined_data.head(10)

# 

# Conitnue to processing RNA data (With Probe Tm)

# Tm Probe 1 (Probe Channel 1)

## Set Constants

1. tmP_inital_T: float (in Celcius) := Inital heating plate temperature for Tm Probes
2. tmP_final_T: float (in Celcius) := Final heating plate temperature for Tm Probes
3. tmP_heating_rate: float (in Celcius/min) := Heating plate temperature rate of change
4. tmP_img_series_gap_time: float (in Seconds) := Time gap between each frame

In [ ]:
print(f"The current Tm Probe 1 Channel is: CY5")

In [ ]:
tmP_1_inital_T = 30
tmP_1_final_T = 85
tmP_1_heating_rate = 10

# For combined Probe1, Probe2 Image stack, used code below to compute tmP_img_series_gap_time
delta_T_tmP_1 = tmP_1_final_T-tmP_1_inital_T
delta_t_tmP_1 = (delta_T_tmP_1/tmP_1_heating_rate)*60
tmP_1_img_series_gap_time = delta_t_tmP_1/(channel_idx_dict["Cy5"][1]-channel_idx_dict["Cy5"][0])
print(tmP_1_img_series_gap_time)

## Select Temperature Range View

#### Note: Temperature range must be a subset within [initial_T, final_T]

In [ ]:
tmP_1_start_temperature = 40
tmP_1_end_temperature = 85

In [ ]:
tmP_1_start_idx, tmP_1_end_idx= utils.select_by_temp_range(temp_range=(tmP_1_start_temperature, tmP_1_end_temperature),
                                                                    initial_T=tmP_1_inital_T,
                                                                    rate_per_min=tmP_1_heating_rate,
                                                                   exposure_in_sec=tmP_1_img_series_gap_time,
                                                                   max_temp=tmP_1_final_T)

In [ ]:
tmP_1_raw_fluorescence_vals = tmP_1_raw_fluorescence_vals_uncut[:, tmP_1_start_idx:tmP_1_end_idx]

In [ ]:
tmP_1_xticks = np.arange(0, tmP_1_raw_fluorescence_vals.shape[1], step=25)
tmP_1_temps = np.zeros(tmP_1_xticks.shape)
for i, frame in enumerate(tmP_1_xticks):
    tmP_1_temps[i] = utils.compute_Tm(initial_T=tmP_1_start_temperature, peak_frame_idx=frame,
                                         rate_per_min=tmP_1_heating_rate, 
                                         exposure_in_sec=tmP_1_img_series_gap_time,
                                        max=tmP_1_end_temperature)
tmP_1_temps = np.round(tmP_1_temps, 1)
tmP_1_temps

## Four levels of Filtering

1. Filter by Tm Barcode_Target Positive
2. Filter by Tm Barcode_Target 2-peak Tm
3. Filter by Shape and start intensity
4. Filter by anomalies occured in imaging (bubbles, etc.)

In [ ]:
importlib.reload(utils)

In [ ]:
# 1. Filter by Tm Barcode_Target Positive
tmP_1_data = tmP_1_raw_fluorescence_vals[positive_idxs][clean_tmB_T_mcs_deriv_mask]
tmP_1_raw_bkg = tmP_1_raw_fluorescence_vals[negative_idxs]
tmP_1_bkg = utils.savgol_filter(utils.min_max_normalize(np.median(tmP_1_raw_bkg, axis=0)),window_length=55, 
                                                             polyorder=2, deriv=0,mode='nearest')
all_tmP_1_positions = positive_pos[clean_tmB_T_mcs_deriv_mask]
# 2. Filter by Tm Barcode_Target 2-peak Tm
tmB_T_corrected_tmP_1_data = tmP_1_data[tmB_T_corrected_idxs]
all_tmP_1_positions = all_tmP_1_positions[tmB_T_corrected_idxs]

In [ ]:
# 3. Filter by Shape and start intensity
tmP_1_masks = utils.interactive_probe_filtering(tmB_T_corrected_tmP_1_data, tmP_1_bkg, mode = "original", max=0.001)

In [ ]:
kmeans_filtered_tmP_1_mask = tmP_1_masks['non_background_mask']
tmP_1_mask_bkg_mask = tmP_1_masks['background_mask']

kmeans_filtered_tmP_1_positive = tmB_T_corrected_tmP_1_data[kmeans_filtered_tmP_1_mask]
kmeans_filtered_tmP_1_negative = tmB_T_corrected_tmP_1_data[tmP_1_mask_bkg_mask]

kmeans_filtered_tmP_1_positive_pos = all_tmP_1_positions[kmeans_filtered_tmP_1_mask]

In [ ]:
kmeans_filtered_tmP_1_positive_pos.shape

In [ ]:
tmP_1_negative_data = tmP_1_raw_fluorescence_vals[negative_idxs]
tmP_1_negative_pos = negative_pos
tmP_1_negative_pos = list(map(tuple, tmP_1_negative_pos))
tmP_1_negative_df = pd.DataFrame({"Pos": tmP_1_negative_pos, **{f"T{i+1}": tmP_1_negative_data[:, i] for i in range(tmP_1_negative_data.shape[1])}})

In [ ]:
tmP_1_positive_pos = list(map(tuple, kmeans_filtered_tmP_1_positive_pos))
tmP_1_positive_df = pd.DataFrame({"Pos": tmP_1_positive_pos, **{f"T{i+1}": kmeans_filtered_tmP_1_positive[:, i] for i in range(kmeans_filtered_tmP_1_positive.shape[1])}})

In [ ]:
importlib.reload(utils)

In [ ]:
tmP_1_positive, QC= utils.subtract_background(tmP_1_positive_df, tmP_1_negative_df, 100)

In [ ]:
tmP_1_positive_arr = tmP_1_positive.iloc[:, 1:].to_numpy()

In [ ]:
kmeans_filtered_tmP_1_positive.shape

## New Background Subtraction Approach for Cy5 Channel Siganls (Tm1):
1. Subtract background within a neighborhood
2. Perform detrending
3. Then move on to taking derivatives, smoothing, & finding peaks

In [ ]:
tmP_1_bkg_subtracted = utils.min_max_normalize(tmP_1_positive_arr, use_global_min_max=True)
normalized_tmP_1_negative = utils.min_max_normalize(kmeans_filtered_tmP_1_negative, use_global_min_max=False)


In [ ]:
tmP_1_mcs_noDeriv = utils.min_max_normalize(1*utils.savgol_filter(tmP_1_bkg_subtracted, window_length=55, 
                                                                      polyorder=2, deriv=0,mode='nearest'), use_global_min_max=True)

tmP_1_bkg_subtracted = utils.min_max_normalize(tmP_1_positive_arr, use_global_min_max=True)



#### Moving Avg. Trend estimation

In [ ]:
trend_est = utils.savgol_filter(tmP_1_bkg_subtracted, window_length=301, 
                                                                      polyorder=2, deriv=0,mode='nearest')

In [ ]:
from scipy.ndimage import gaussian_filter1d
def estimate_trend_with_gaussian_smoothing(signals, sigma=2, radius = 10):
    smoothed_trends = gaussian_filter1d(signals, sigma=sigma, axis=1, radius = radius, mode = "nearest")
    return smoothed_trends

In [ ]:
trend_est = estimate_trend_with_gaussian_smoothing(tmP_1_bkg_subtracted, radius = 150, sigma=30)

In [ ]:
tmP_1_bkg_subtracted_detrended = tmP_1_bkg_subtracted-trend_est

In [ ]:
second_trend_est = estimate_trend_with_gaussian_smoothing(tmP_1_bkg_subtracted_detrended, radius = 200, sigma=30)

In [ ]:
tmP_1_bkg_subtracted_detrended = tmP_1_bkg_subtracted_detrended-second_trend_est

In [ ]:
tmP_1_bkg_subtracted_detrended = utils.min_max_normalize(tmP_1_bkg_subtracted_detrended, use_global_min_max=False)

In [ ]:
tmP_1_mcs_deriv, tmP_1_min_max= utils.min_max_normalize(-1*utils.savgol_filter(tmP_1_bkg_subtracted_detrended, window_length=55, 
                                                                     polyorder=2, deriv=1,mode='nearest'),use_global_min_max=True, 
                                                                      return_min_max= True)

In [ ]:
tmP_1_negative_mcs_deriv = utils.min_max_normalize(-1*utils.savgol_filter(normalized_tmP_1_negative, window_length=55, 
                                                                     polyorder=2, deriv=1,mode='nearest'),use_global_min_max=True,
                                                      use_predefined_min_max_param=True, predefined_min=tmP_1_min_max[0],
                                                      predefined_max=tmP_1_min_max[1])

In [ ]:
importlib.reload(utils)

In [ ]:
tmP_1_mcs_deriv_smoothed = utils.gaussian_smooth(tmP_1_mcs_deriv, sigma = 10)

In [ ]:
utils.visualize_background_subtraction_qc(QC, tmP_1_positive_df, tmP_1_negative_df, tmP_1_positive,
                                          tmP_1_bkg_subtracted_detrended, additional_array=tmP_1_mcs_deriv_smoothed, corner_threshold=200,
                                   n_corner_points_to_sample=5, n_central_points_to_sample=5)

In [ ]:
raw_normalized_tmP_1 = utils.min_max_normalize(tmP_1_positive_arr, use_global_min_max=True)
normalized_tmP_1_negative = utils.min_max_normalize(kmeans_filtered_tmP_1_negative, use_global_min_max=False)

In [ ]:
# 4. Filter by anomalies occured in imaging (bubbles, etc.)
tmP_1_anomaly_mask = utils.interactive_anomaly_filtering(raw_normalized_tmP_1, mode="iso", max = 0.5)

In [ ]:
anomaly_filtered_tmP_1_mask = tmP_1_anomaly_mask['normal_mask']
tmP_1_abnormal_mask = tmP_1_anomaly_mask['anomaly_mask']

normalized_tmP_1 = raw_normalized_tmP_1[anomaly_filtered_tmP_1_mask]
filtered_tmP_1_abnormal = raw_normalized_tmP_1[tmP_1_abnormal_mask]

In [ ]:
fig_width = 15
fig_height = 5
plt.figure(figsize=(fig_width, fig_height))

plt.subplot(1, 2, 1) 
for signal in normalized_tmP_1:
    plt.plot(signal)
plt.xlabel("Temperature")
plt.xticks(ticks=tmP_1_xticks, labels=tmP_1_temps)
plt.ylabel("Fluorescence Value")
plt.title("Normalized Tm Probe 1 Positive")

plt.subplot(1, 2, 2)
for signal in normalized_tmP_1_negative:
    plt.plot(signal)
plt.xticks(ticks=tmP_1_xticks, labels=tmP_1_temps)
# plt.plot(tmP_1_bkg*tmP_1_bkg_strength, color = "black", label="Strength Adjusted Background", linestyle="dotted", linewidth=2)
plt.xlabel("Temperature")
plt.ylabel("Normalized Fluorescence Value")
plt.title("Normalized Tm Probe Negative")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# normalized_tmP_1 = utils.min_max_normalize(tmB_T_corrected_tmP_1_data, use_global_min_max=False)

In [ ]:
fig_width = 15
fig_height = 5
plt.figure(figsize=(fig_width, fig_height))

plt.subplot(1, 2, 1) 
for signal in tmB_T_corrected_tmP_1_data:
    plt.plot(signal)
plt.xlabel("Frames")
plt.ylabel("Fluorescence Value")
plt.title("Tm Probe 1 with background")

plt.subplot(1, 2, 2)
for signal in normalized_tmP_1:
    plt.plot(signal)
plt.xticks(ticks=tmP_1_xticks, labels=tmP_1_temps)
# plt.plot(tmP_1_bkg, color = "black", label="Strength Adjusted Background", linestyle="dotted", linewidth=2)
plt.xlabel("Temperature")
plt.ylabel("Normalized Fluorescence Value")
plt.title("Normalized Tm Probe 1 (Background Subtracted)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Save raw background substracted Cy5 (Tm1) Signal
np.save("bkg_subtracted_raw_"+tmP_1_save_filename, tmP_1_positive_arr)

## Compute Tm Probe 1

#### Important Note: When necessary, you can flip the melting Curve to help computing Tm

In [ ]:
importlib.reload(utils)

In [ ]:
flip = False
if flip:
    tmP_1_mcs_deriv = utils.min_max_normalize(-1*tmP_1_mcs_deriv, use_global_min_max=True)
else:
    tmP_1_mcs_deriv = utils.min_max_normalize(1*tmP_1_mcs_deriv, use_global_min_max=True)

#### Note: Set flat_noise to false to estimate a noise trend

In [ ]:
importlib.reload(utils)

### Updates here: allows a list of prominence arrays as input for each cluster

In [ ]:
tmP_1_peak_prominence_arr_1 = utils.generate_variable_threshold(signal_length=tmP_1_mcs_deriv.shape[1],
                                                           low_threshold_segments=[(100,125),(200,260)],
                                                           low_threshold_value=.1,
                                                           high_threshold_value=1)
tmP_1_peak_prominence_arr_2 = utils.generate_variable_threshold(signal_length=tmP_1_mcs_deriv.shape[1],
                                                           low_threshold_segments=[(100,125),(200,260)],
                                                           low_threshold_value=0.3,
                                                           high_threshold_value=1)
tmP_1_peak_prominence_arr_3 = utils.generate_variable_threshold(signal_length=tmP_1_mcs_deriv.shape[1],
                                                           low_threshold_segments=[(100,125),(200,260)],
                                                           low_threshold_value=0.5,
                                                           high_threshold_value=1)
tmP_1_peak_prominence_arr_4 = utils.generate_variable_threshold(signal_length=tmP_1_mcs_deriv.shape[1],
                                                           low_threshold_segments=[(100,125),(200,260)],
                                                           low_threshold_value=0.3,
                                                           high_threshold_value=1)

In [ ]:
all_tmP_1s, all_tmP_1_n_SDs, all_tmP_1_heights = utils.interactive_probe_clustering_thresholding(tmP_1_mcs_deriv_smoothed, 
                                                                                                              max_clusters=4, 
                                                                                n_SD=[0,0,0,0],                      
                                                                               initial_T=tmP_1_start_temperature, 
                                                                                final_T=tmP_1_end_temperature,
                                                                               heating_rate=tmP_1_heating_rate, img_series_gap_time=tmP_1_img_series_gap_time,
                                                                            normalized_negatives=tmP_1_negative_mcs_deriv,
                                                                                flat_noise=[True, True, True, True], width_range = (5),
                                                                                use_negative_est_SD=True,
                            prominance_array=[tmP_1_peak_prominence_arr_1, tmP_1_peak_prominence_arr_2,
                                             tmP_1_peak_prominence_arr_3,tmP_1_peak_prominence_arr_4],
                                                                                baseline_offset=[True]*4, mannual_offset_fitting_range=(200,270))

In [ ]:
importlib.reload(utils)

## Visual Verification for peak finding

In [ ]:
utils.plot_individual_probe_signal(tmP_1_mcs_deriv_smoothed, all_tmP_1s,
                                  initial_T=tmP_1_start_temperature, final_T=tmP_1_end_temperature,
                heating_rate=tmP_1_heating_rate, img_series_gap_time=tmP_1_img_series_gap_time,raw_data_array=None)

In [ ]:
# 1. Filter by Tm Barcode_Target Positive
tmP_1_data = tmP_1_raw_fluorescence_vals[positive_idxs][clean_tmB_T_mcs_deriv_mask]
tmP_1_raw_bkg = tmP_1_raw_fluorescence_vals[negative_idxs]
tmP_1_bkg = utils.savgol_filter(utils.min_max_normalize(np.median(tmP_1_raw_bkg, axis=0)),window_length=55, 
                                                             polyorder=2, deriv=0,mode='nearest')
all_tmP_1_positions = positive_pos[clean_tmB_T_mcs_deriv_mask]
# 2. Filter by Tm Barcode_Target 2-peak Tm
tmB_T_corrected_tmP_1_data = tmP_1_data[tmB_T_corrected_idxs]
all_tmP_1_positions = all_tmP_1_positions[tmB_T_corrected_idxs]
kmeans_filtered_tmP_1_mask = tmP_1_masks['non_background_mask']
tmP_1_mask_bkg_mask = tmP_1_masks['background_mask']

kmeans_filtered_tmP_1_positive = tmB_T_corrected_tmP_1_data[kmeans_filtered_tmP_1_mask]
kmeans_filtered_tmP_1_negative = tmB_T_corrected_tmP_1_data[tmP_1_mask_bkg_mask]

kmeans_filtered_tmP_1_positive_pos = all_tmP_1_positions[kmeans_filtered_tmP_1_mask]

In [ ]:
expected_tmP_1_values = [57, 78.]
max_number_of_tmP_1s_per_signal = 2
filtered_tmP_1s, encoded_tmP_1s = utils.filter_local_tms(all_tmP_1s,
                                             expected_tm_values=expected_tmP_1_values, resolution=5, 
                                             peak_heights=all_tmP_1_heights, max_n_tms=max_number_of_tmP_1s_per_signal)

In [ ]:
re_filled_filtered_tmP_1s = np.full((tmB_T_corrected_tmP_1_data.shape[0],2*max_number_of_tmP_1s_per_signal), 
                                        (max_number_of_tmP_1s_per_signal*[np.nan]+max_number_of_tmP_1s_per_signal*[0]))
re_filled_filtered_tmP_1s_encode = np.full((tmB_T_corrected_tmP_1_data.shape[0],2*max_number_of_tmP_1s_per_signal), 
                                        (max_number_of_tmP_1s_per_signal*[np.nan]+max_number_of_tmP_1s_per_signal*[0]))
refilled_confidence_level_tmP_1s = np.full((tmB_T_corrected_tmP_1_data.shape[0]), 0)

filtered_indices_first_mask = np.where(kmeans_filtered_tmP_1_mask)[0]
filtered_indices_second_mask = filtered_indices_first_mask[anomaly_filtered_tmP_1_mask]

re_filled_filtered_tmP_1s[filtered_indices_second_mask] = filtered_tmP_1s
re_filled_filtered_tmP_1s_encode[filtered_indices_second_mask] = encoded_tmP_1s

refilled_confidence_level_tmP_1s[filtered_indices_second_mask] = all_tmP_1_n_SDs

refilled_tmP_1s_data = np.full((tmB_T_corrected_tmP_1_data.shape[0], normalized_tmP_1.shape[1]), np.nan)
refilled_tmP_1s_data[filtered_indices_second_mask] = tmP_1_mcs_deriv

refilled_tmP_1s_data_noDerive = np.full((tmB_T_corrected_tmP_1_data.shape[0], normalized_tmP_1.shape[1]), np.nan)
refilled_tmP_1s_data_noDerive[filtered_indices_second_mask] = tmP_1_mcs_noDeriv

In [ ]:
np.save("dfc_"+tmP_1_save_filename, refilled_tmP_1s_data)

- CY5: 77.6, 57.6; RES: +-1
- HEX: 67, 43; RES: +-1

### Plot Tm Distribution

In [ ]:
importlib.reload(utils)

In [ ]:
utils.visualize_local_tms_distribution(re_filled_filtered_tmP_1s, expected_tm_values=expected_tmP_1_values, 
                                       resolution = 1., max_n_tms=max_number_of_tmP_1s_per_signal)

# Tm Probe 2 (Probe Channel 2)

## Set Constants

1. tmP_inital_T: float (in Celcius) := Inital heating plate temperature for Tm Probes
2. tmP_final_T: float (in Celcius) := Final heating plate temperature for Tm Probes
3. tmP_heating_rate: float (in Celcius/min) := Heating plate temperature rate of change
4. tmP_img_series_gap_time: float (in Seconds) := Time gap between each frame

In [ ]:
print(f"The current Tm Probe 2 Channel is: DsRed-hex")

In [ ]:
tmP_2_inital_T = 30.0
tmP_2_final_T = 85.0
tmP_2_heating_rate = 10

# For combined Probe1, Probe2 Image stack, used code below to compute tmP_img_series_gap_time
delta_T_tmP_2 = tmP_2_final_T-tmP_2_inital_T
delta_t_tmP_2 = (delta_T_tmP_2/tmP_2_heating_rate)*60
tmP_2_img_series_gap_time = delta_t_tmP_2/(channel_idx_dict["HEX"][1]-channel_idx_dict["HEX"][0])
print(tmP_2_img_series_gap_time)

In [ ]:
tmP_2_start_temperature = 30.0
tmP_2_end_temperature = 85.0
tmP_2_start_idx, tmP_2_end_idx= utils.select_by_temp_range(temp_range=(tmP_2_start_temperature, tmP_2_end_temperature),
                                                                    initial_T=tmP_2_inital_T,
                                                                    rate_per_min=tmP_2_heating_rate,
                                                                   exposure_in_sec=tmP_2_img_series_gap_time,
                                                                   max_temp=tmP_2_final_T)
tmP_2_raw_fluorescence_vals = tmP_2_raw_fluorescence_vals_uncut[:, tmP_2_start_idx:tmP_2_end_idx]


tmP_2_xticks = np.arange(0, tmP_2_raw_fluorescence_vals.shape[1], step=25)
tmP_2_temps = np.zeros(tmP_2_xticks.shape)
for i, frame in enumerate(tmP_2_xticks):
    tmP_2_temps[i] = utils.compute_Tm(initial_T=tmP_2_start_temperature, peak_frame_idx=frame,
                                         rate_per_min=tmP_2_heating_rate, 
                                         exposure_in_sec=tmP_2_img_series_gap_time,
                                        max=tmP_2_end_temperature)
tmP_2_temps = np.round(tmP_2_temps, 1)
tmP_2_temps

## Four levels of Filtering

1. Filter by Tm Barcode_Target Positive
2. Filter by Tm Barcode_Target 2-peak Tm
3. Filter by Start intensity
4. Filter by anomalies occured in imaging (bubbles, etc.)

In [ ]:
importlib.reload(utils)

In [ ]:
# 1.
tmP_2_data = tmP_2_raw_fluorescence_vals[positive_idxs][clean_tmB_T_mcs_deriv_mask]
tmP_2_raw_bkg = tmP_2_raw_fluorescence_vals[negative_idxs]
tmP_2_bkg = utils.savgol_filter(utils.min_max_normalize(np.median(tmP_2_raw_bkg, axis=0)),window_length=55, 
                                                             polyorder=2, deriv=0,mode='nearest')

all_tmP_2_positions = positive_pos[clean_tmB_T_mcs_deriv_mask]

# 2.
tmB_T_corrected_tmP_2_data = tmP_2_data[tmB_T_corrected_idxs]
all_tmP_2_positions = all_tmP_2_positions[tmB_T_corrected_idxs]

In [ ]:
# 1.
tmP_2_start_intensity = probe_first_frame_intensities[1][positive_idxs]
# 2.
tmP_2_start_intensity = tmP_2_start_intensity[tmB_T_corrected_idxs]


In [ ]:
tmP_2_masks = utils.interactive_probe_filtering(tmB_T_corrected_tmP_2_data, tmP_2_bkg, mode = "original",max=0.001)

In [ ]:
kmeans_filtered_tmP_2_mask = tmP_2_masks['non_background_mask']
tmP_2_mask_bkg_mask = tmP_2_masks['background_mask']

kmeans_filtered_tmP_2_positive = tmB_T_corrected_tmP_2_data[kmeans_filtered_tmP_2_mask]
kmeans_filtered_tmP_2_negative = tmB_T_corrected_tmP_2_data[tmP_2_mask_bkg_mask]
kmeans_filtered_tmP_2_positive_pos = all_tmP_2_positions[kmeans_filtered_tmP_2_mask]

In [ ]:
tmP_2_negative_data = tmP_2_raw_fluorescence_vals[negative_idxs]
tmP_2_negative_pos = negative_pos
tmP_2_negative_pos = list(map(tuple, tmP_2_negative_pos))
tmP_2_negative_df = pd.DataFrame({"Pos": tmP_2_negative_pos, **{f"T{i+1}": tmP_2_negative_data[:, i] for i in range(tmP_2_negative_data.shape[1])}})


tmP_2_positive_pos = list(map(tuple, kmeans_filtered_tmP_2_positive_pos))
tmP_2_positive_df = pd.DataFrame({"Pos": tmP_2_positive_pos, **{f"T{i+1}": kmeans_filtered_tmP_2_positive[:, i] for i in range(kmeans_filtered_tmP_2_positive.shape[1])}})

In [ ]:
tmP_2_positive, QC= utils.subtract_background(tmP_2_positive_df, tmP_2_negative_df, 100)

In [ ]:
tmP_2_positive_arr = tmP_2_positive.iloc[:, 1:].to_numpy()

In [ ]:
tmP_2_positive_arr.shape

In [ ]:
tmB_T_corrected_tmP_2_data.shape

## New Background Subtraction Approach for HEX Channel Siganls (Tm2):
1. Subtract background within a neighborhood
2. Then move on to taking derivatives, smoothing, & finding peaks

In [ ]:
tmP_2_bkg_subtracted = utils.min_max_normalize(tmP_2_positive_arr, use_global_min_max=True)

tmP_2_mcs_deriv, tmP_2_min_max= utils.min_max_normalize(-1*utils.savgol_filter(tmP_2_bkg_subtracted, window_length=55, 
                                                                     polyorder=2, deriv=1,mode='nearest'),use_global_min_max=True, 
                                                                      return_min_max= True)
tmP_2_mcs_noDeriv = utils.min_max_normalize(1*utils.savgol_filter(tmP_2_bkg_subtracted, window_length=55, 
                                                                      polyorder=2, deriv=0,mode='nearest'), use_global_min_max=True)

In [ ]:
normalized_tmP_2_negative = utils.min_max_normalize(kmeans_filtered_tmP_2_negative, use_global_min_max=False)
tmP_2_negative_mcs_deriv = utils.min_max_normalize(-1*utils.savgol_filter(normalized_tmP_2_negative, window_length=55, 
                                                                     polyorder=2, deriv=1,mode='nearest'),use_global_min_max=True,
                                                      use_predefined_min_max_param=True, predefined_min=tmP_2_min_max[0],
                                                      predefined_max=tmP_2_min_max[1])

In [ ]:
utils.visualize_background_subtraction_qc(QC, tmP_2_positive_df, tmP_2_negative_df, tmP_2_positive,
                                          tmP_2_bkg_subtracted, additional_array=tmP_2_mcs_deriv, corner_threshold=200,
                                   n_corner_points_to_sample=8, n_central_points_to_sample=12)

In [ ]:
raw_normalized_tmP_2 = utils.min_max_normalize(tmP_2_positive_arr, use_global_min_max=True)
normalized_tmP_2_negative = utils.min_max_normalize(kmeans_filtered_tmP_2_negative, use_global_min_max=False)

In [ ]:
tmP_2_anomaly_mask = utils.interactive_anomaly_filtering(raw_normalized_tmP_2, mode="iso")

In [ ]:
anomaly_filtered_tmP_2_mask = tmP_2_anomaly_mask['normal_mask']
tmP_2_abnormal = tmP_2_anomaly_mask['anomaly_mask']

normalized_tmP_2 = raw_normalized_tmP_2[anomaly_filtered_tmP_2_mask]
filtered_tmP_2_abnormal = raw_normalized_tmP_2[tmP_2_abnormal]

In [ ]:
fig_width = 15
fig_height = 5
plt.figure(figsize=(fig_width, fig_height))

plt.subplot(1, 2, 1) 
for signal in normalized_tmP_2:
    plt.plot(signal)
plt.xlabel("Frames")
plt.ylabel("Fluorescence Value")
plt.title("Normalized Tm Probe 2 Positives")

plt.subplot(1, 2, 2)
for signal in normalized_tmP_2_negative:
    plt.plot(signal)
plt.xticks(ticks=tmP_2_xticks, labels=tmP_2_temps)
plt.xlabel("Temperature")
plt.ylabel("Normalized Fluorescence Value")
plt.title("Normalized Tm Probe 2 Negatives")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig_width = 15
fig_height = 5
plt.figure(figsize=(fig_width, fig_height))

plt.subplot(1, 2, 1) 
for signal in tmB_T_corrected_tmP_2_data:
    plt.plot(signal)
plt.xlabel("Frames")
plt.ylabel("Fluorescence Value")
plt.title("Tm Probe 2 with background")

plt.subplot(1, 2, 2)
for signal in normalized_tmP_2:
    plt.plot(signal)
plt.xticks(ticks=tmP_2_xticks, labels=tmP_2_temps)
plt.xlabel("Temperature")
plt.ylabel("Normalized Fluorescence Value")
plt.title("Normalized Tm Probe 2 (Bakcgroudn Subtracted)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Save raw background substracted Cy5 (Tm2) Signal
np.save("bkg_subtracted_raw_"+tmP_2_save_filename, tmP_2_positive_arr)

## Compute Tm Probe 2

#### Important Note: When necessary, you can flip the melting Curve to help computing Tm

In [ ]:
flip = False
if flip:
    tmP_2_mcs_deriv = utils.min_max_normalize(-1*tmP_2_mcs_deriv, use_global_min_max=True)
else:
    tmP_2_mcs_deriv = utils.min_max_normalize(tmP_2_mcs_deriv, use_global_min_max=True)

#### Note: Set flat_noise to false to get noise trend estimation

In [ ]:
importlib.reload(utils)

In [ ]:
tmP_2_peak_prominence_arr_1 = utils.generate_variable_threshold(signal_length=normalized_tmP_2.shape[1],
                                                           low_threshold_segments=[(45,125)],
                                                           low_threshold_value=.05,
                                                           high_threshold_value=1)

tmP_2_peak_prominence_arr_2 = utils.generate_variable_threshold(signal_length=normalized_tmP_2.shape[1],
                                                           low_threshold_segments=[(200,240)],
                                                           low_threshold_value=.5,
                                                           high_threshold_value=1)

In [ ]:
tmP_2_mcs_deriv_smoothed = utils.gaussian_smooth(tmP_2_mcs_deriv, sigma = 10)

In [ ]:
all_tmP_2s, all_tmP_2_n_SDs, all_tmP_2_heights = utils.interactive_probe_clustering_thresholding(tmP_2_mcs_deriv_smoothed, 
                                                                                                              max_clusters=2, 
                                                                                n_SD = [0.3,1.5], 
                                                                               initial_T=tmP_2_start_temperature, 
                                                                                final_T=tmP_2_end_temperature,
                                                                               heating_rate=tmP_2_heating_rate, img_series_gap_time=tmP_2_img_series_gap_time,
                                                                                flat_noise=[True, True], use_negative_est_SD=False,
                                                                                normalized_negatives = tmP_2_negative_mcs_deriv,
                                                        prominance_array=[tmP_2_peak_prominence_arr_1, tmP_2_peak_prominence_arr_2],
                                                                                baseline_offset=[True, True], 
                                                                                mannual_offset_fitting_range=None)
                                                                                                             

In [ ]:
importlib.reload(utils)

In [ ]:
utils.plot_individual_probe_signal(tmP_2_mcs_deriv_smoothed, all_tmP_2s,
                                  initial_T=tmP_2_start_temperature, final_T=tmP_2_end_temperature,
                heating_rate=tmP_2_heating_rate, img_series_gap_time=tmP_2_img_series_gap_time,raw_data_array=None, y_lim=(0., 1))

In [ ]:
expected_tmP_2_values = [43, 67]
max_number_of_tmP_2s_per_signal = 2
filtered_tmP_2s, encoded_tmP_2s = utils.filter_local_tms(all_tmP_2s,
                                             expected_tm_values=expected_tmP_2_values, resolution=5, 
                                            peak_heights=all_tmP_2_heights, max_n_tms=max_number_of_tmP_2s_per_signal)

In [ ]:
re_filled_filtered_tmP_2s = np.full((tmB_T_corrected_tmP_2_data.shape[0],2*max_number_of_tmP_2s_per_signal), 
                                        (max_number_of_tmP_2s_per_signal*[np.nan]+max_number_of_tmP_2s_per_signal*[0]))
re_filled_filtered_tmP_2s_encode = np.full((tmB_T_corrected_tmP_2_data.shape[0],2*max_number_of_tmP_2s_per_signal), 
                                        (max_number_of_tmP_2s_per_signal*[np.nan]+max_number_of_tmP_2s_per_signal*[0]))
refilled_confidence_level_tmP_2s = np.full((tmB_T_corrected_tmP_2_data.shape[0]), 0)

refilled_tmP_2s_data = np.full((tmB_T_corrected_tmP_2_data.shape[0], 
                                    normalized_tmP_2.shape[1]), np.nan)

filtered_indices_first_mask_tmP_2 = np.where(kmeans_filtered_tmP_2_mask)[0]
filtered_indices_second_mask_tmP_2 = filtered_indices_first_mask_tmP_2[anomaly_filtered_tmP_2_mask]

re_filled_filtered_tmP_2s[filtered_indices_second_mask_tmP_2] = filtered_tmP_2s
re_filled_filtered_tmP_2s_encode[filtered_indices_second_mask_tmP_2] = encoded_tmP_2s
refilled_confidence_level_tmP_2s[filtered_indices_second_mask_tmP_2] = all_tmP_2_n_SDs

refilled_tmP_2s_data[filtered_indices_second_mask_tmP_2] = tmP_2_mcs_deriv


refilled_tmP_2s_data_noDerive = np.full((tmB_T_corrected_tmP_2_data.shape[0], normalized_tmP_2.shape[1]), np.nan)
refilled_tmP_2s_data_noDerive[filtered_indices_second_mask_tmP_2] = tmP_2_mcs_noDeriv

In [ ]:
np.save("dfc_"+tmP_2_save_filename, refilled_tmP_2s_data)

In [ ]:
tmP_2_save_filename

### Plot Tm Distribution

In [ ]:
importlib.reload(utils)

In [ ]:
utils.visualize_local_tms_distribution(re_filled_filtered_tmP_2s, expected_tm_values=expected_tmP_2_values,
                                       resolution = 2, max_n_tms=max_number_of_tmP_2s_per_signal)

# Combine All Tm Information For Clustering

In [ ]:
importlib.reload(utils)

In [ ]:
probe_channel_names = ['Cy5', 'DsRed-hex']
combined_tms_encoded_no_encode = utils.join_all_tms(parsed_local_tms_list = [re_filled_filtered_tmP_1s_encode, re_filled_filtered_tmP_2s_encode], 
                                          global_tms = tmB_Ts, 
                                          max_n_tms_list = [max_number_of_tmP_1s_per_signal, max_number_of_tmP_2s_per_signal],
                                          ordered_channel_names= probe_channel_names, 
                                          expected_tm_vals=[expected_tmP_1_values, expected_tmP_2_values],
                          keep_nonSpecific=False, output_DataFrame=True, encoding=False)

In [ ]:
probe_channel_names = ['Cy5', 'DsRed-hex']
combined_tms_encoded = utils.join_all_tms(parsed_local_tms_list = [re_filled_filtered_tmP_1s_encode, re_filled_filtered_tmP_2s_encode], 
                                          global_tms = tmB_Ts, 
                                          max_n_tms_list = [max_number_of_tmP_1s_per_signal, max_number_of_tmP_2s_per_signal],
                                          ordered_channel_names= probe_channel_names, 
                                          expected_tm_vals=[expected_tmP_1_values, expected_tmP_2_values],
                          keep_nonSpecific=False, output_DataFrame=True, encoding=True)

In [ ]:
combined_tms_encoded.head()

In [ ]:
combined_tms_encoded.rename(columns={"Global Tm 1": "LowTm", "Global Tm 2": "HighTm"}, inplace=True)
combined_tms_encoded_no_encode.rename(columns={"Global Tm 1": "LowTm", "Global Tm 2": "HighTm"}, inplace=True)

### Incorporate Position, Fluorescence Info

In [ ]:
importlib.reload(utils)

In [ ]:
filtered_pos = positive_pos[clean_tmB_T_mcs_deriv_mask][tmB_T_corrected_idxs]

filtered_raw_fluorescence = tmB_T_data[clean_tmB_T_mcs_deriv_mask][tmB_T_corrected_idxs]

tmB_T_bkg_subtracted_final = tmB_T_bkg_subtracted[tmB_T_corrected_idxs]

In [ ]:
all_combined = utils.join_meta_data(combined_tms_encoded, filtered_pos, 
                                    filtered_raw_fluorescence, [refilled_confidence_level_tmP_1s,refilled_confidence_level_tmP_2s])

In [ ]:
all_combined_no_encode = utils.join_meta_data(combined_tms_encoded_no_encode, filtered_pos, 
                                    filtered_raw_fluorescence, [refilled_confidence_level_tmP_1s,refilled_confidence_level_tmP_2s])

In [ ]:
all_combined_bkg_subtracted = utils.join_meta_data(combined_tms_encoded, filtered_pos, 
                                    tmB_T_bkg_subtracted_final, [refilled_confidence_level_tmP_1s,refilled_confidence_level_tmP_2s])

In [ ]:
tmB_T_derive = reduced_window_derive[tmB_T_corrected_idxs]
all_combined_derive = utils.join_meta_data(combined_tms_encoded, filtered_pos, 
                                    tmB_T_derive, [refilled_confidence_level_tmP_1s,refilled_confidence_level_tmP_2s])

In [ ]:
all_combined_no_encode.head()

In [ ]:
all_combined_bkg_subtracted.to_csv(r"raw_tm_result_bkg_subtracted.csv")

In [ ]:
all_combined_no_encode.to_csv(r"raw_tm_result_no_encode.csv")

In [ ]:
all_combined.to_csv(r"raw_tm_result.csv")

In [ ]:
tmP_1_save_filename

In [ ]:
tmP_1_combined = utils.join_meta_data(combined_tms_encoded, filtered_pos, 
                                    refilled_tmP_1s_data_noDerive)
tmP_2_combined = utils.join_meta_data(combined_tms_encoded, filtered_pos, 
                                    refilled_tmP_2s_data_noDerive)
tmP_1_combined.to_csv(tmP_1_save_filename+"tm_profile_probe1_intensity.csv", index=False)
tmP_2_combined.to_csv(tmP_2_save_filename+r"tm_profile_probe2_intensity.csv", index=False)

In [ ]:
all_combined = pd.read_csv("raw_tm_result.csv", index_col=0)

In [ ]:
all_combined = all_combined_derive

In [ ]:
# all_combined = all_combined_bkg_subtracted

In [ ]:
column_pairs = [('Cy5 Tm 1', 'Cy5 Tm 2'), ('DsRed-hex Tm 1', 'DsRed-hex Tm 2')]
all_combined = utils.shift_nonzero_to_first(all_combined, column_pairs)

In [ ]:
all_combined.head()

In [ ]:
raw_tmP_col_names = ["Raw_Cy5_1","Raw_Cy5_2", "Raw_Hex_1", "Raw_Hex_2"]
tm_col_names = ['Cy5 Tm 1', 'Cy5 Tm 2', 'DsRed-hex Tm 1','DsRed-hex Tm 2']
all_combined[raw_tmP_col_names] = all_combined_no_encode[tm_col_names]

### Section Below is for checking all Nones across probe channels. (Please skip)

In [ ]:
all_none_idxs = all_combined[(all_combined[['Cy5 Tm 1', 'Cy5 Tm 2', 'DsRed-hex Tm 1', 'DsRed-hex Tm 2']] == 0).all(axis=1)].index.tolist()

In [ ]:
all_combined.loc[all_none_idxs].shape

In [ ]:
# Load derivative data
re_filled_filtered_tmP_1_mcs_deriv = np.full((tmB_T_corrected_tmP_1_data.shape[0],tmP_1_mcs_deriv.shape[1]), np.nan)
re_filled_filtered_tmP_1_mcs_deriv[filtered_indices_second_mask] = tmP_1_mcs_deriv
re_filled_filtered_tmP_2_mcs_deriv = np.full((tmB_T_corrected_tmP_2_data.shape[0],tmP_2_mcs_deriv.shape[1]), np.nan)
re_filled_filtered_tmP_2_mcs_deriv[filtered_indices_second_mask_tmP_2] = tmP_2_mcs_deriv

In [ ]:
# Load raw data if needed
re_filled_filtered_tmP_1_mcs_deriv[filtered_indices_second_mask] = normalized_tmP_1
re_filled_filtered_tmP_2_mcs_deriv[filtered_indices_second_mask_tmP_2] = normalized_tmP_2

In [ ]:
N_tmP_1_nonSpecific = np.sum(~np.any(np.isnan(re_filled_filtered_tmP_1_mcs_deriv[all_none_idxs]), axis=1))
N_tmP_2_nonSpecific = np.sum(~np.any(np.isnan(re_filled_filtered_tmP_2_mcs_deriv[all_none_idxs]), axis=1))
N_tot_allNones = np.sum(~np.any(np.isnan(tmB_T_mcs_deriv[tmB_T_corrected_idxs][all_none_idxs]), axis=1))

In [ ]:
print(f"Total number of all Nones is {N_tot_allNones}. There are {N_tmP_1_nonSpecific} non-specific Tms from Tm Probe Channel 1, and {N_tmP_2_nonSpecific} non-specific Tms from Tm Probe Channel 2 ")

### Ready for clustering! (Resume here)

In [ ]:
importlib.reload(utils)

In [ ]:
subset_dfs = utils.split_dataframe_by_columns(all_combined,
                                              columns=['Cy5 Tm 1', 'DsRed-hex Tm 1',])
subset_dfs_bkg_subtracted = utils.split_dataframe_by_columns(all_combined_bkg_subtracted, 
                                                             columns=['Cy5 Tm 1', 'DsRed-hex Tm 1'])

In [ ]:
# (Tm tuple, channel key name)
tm_key_pairs = [
    ((expected_tmP_1_values[1], 0.0), "B1R1"),
    ((0.0, expected_tmP_2_values[1]), "B1G1"),
    ((expected_tmP_1_values[0], expected_tmP_2_values[1]), "R1G1"),
    ((expected_tmP_1_values[1], expected_tmP_2_values[0]), "G1R1")
]

valid_lib_keys_dict = dict(tm_key_pairs)
key_to_tm_mapping = {label: tm for tm, label in tm_key_pairs}

valid_subsets = {label: subset_dfs[tm] for tm, label in tm_key_pairs if tm in subset_dfs}
valid_subsets_bkg_subtracted = {
    label + "_bkg_subtracted": subset_dfs_bkg_subtracted[tm]
    for tm, label in tm_key_pairs
    if tm in subset_dfs_bkg_subtracted
}
encoded_subsets = valid_subsets
encoded_subsets_bkg_subtracted = valid_subsets_bkg_subtracted

In [ ]:
sorted_keys = sorted(subset_dfs.keys(), key=lambda k: subset_dfs[k].shape[0], reverse=True)

for key in sorted_keys:
    print(key, subset_dfs[key].shape)

In [ ]:
tm_key_pairs

In [ ]:
utils.save_subset_dfs(encoded_subsets_bkg_subtracted)

In [ ]:
utils.save_subset_dfs(encoded_subsets)

In [ ]:
utils.plot_subset_scatter(valid_subsets, col_names = ["LowTm","HighTm"], 
                          channel_names = ['Cy5 Tm 1', 'DsRed-hex Tm 1'], 
                          key_to_tm_mapping = key_to_tm_mapping)